## Exercise 5

In [3]:
# Part 1 
import scipy.stats as stats
import numpy as np
# estimate integral, by simulating using the crude monte carlo
def compute_ci(data, alpha=0.05):
    """Calculates the 95% Confidence Interval based on sub-samples."""
    n = len(data)
    mean = np.mean(data)
    std_err = np.std(data, ddof=1) / np.sqrt(n)
    t_crit = stats.t.ppf(1 - alpha/2, df=n - 1)
    return mean, mean - t_crit * std_err, mean + t_crit * std_err

# int(0,1) (exp(x))
np.random.seed(42)

import numpy as np
sample_size = 100

# from slide 13 day 4

U = np.random.uniform(0,1, size=sample_size)

Y = np.exp(U)
theta_monte_carlo = np.mean(Y)
mean, lower, upper = compute_ci(Y)
print(f'Estimated integral with mothe carlo n={sample_size} is {np.round(theta_monte_carlo,4)}')
print(f"mean: {np.round(mean,4)}, Confidence 95%: [{np.round(lower,4)}, {np.round(upper,4)}] ")

Estimated integral with mothe carlo n=100 is 1.6721
mean: 1.6721, Confidence 95%: [1.5728, 1.7713] 


In [4]:
# Part 2 estimate integral int(0,1) (exp(x))
# using antithetic variables 
np.random.seed(42)
U = np.random.uniform(0,1, size=sample_size)
Y = (np.exp(U) + np.exp(1-U))/2
antithetic_theta = np.mean(Y)


mean, lower, upper = compute_ci(Y)
print(f'Estimated integral with antithetic variables n={sample_size} is {np.round(antithetic_theta,4)}')
print(f"mean: {np.round(mean,4)}, Confidence 95%: [{np.round(lower,4)}, {np.round(upper,4)}] ")

Estimated integral with antithetic variables n=100 is 1.7226
mean: 1.7226, Confidence 95%: [1.7102, 1.735] 


In [5]:
# part 3 estimate integral int(0,1) (exp(x))
# usin a control variate and comåarable computational resources
np.random.seed(42)
U = np.random.uniform(0,1, size=sample_size)
Z = U
X = np.exp(U)
E_muz = 0.5
varz = np.var(Z)
cov = np.cov(U,X)
c = -cov[0,1]/varz
 
Y =  X + c*(Z-E_muz)
control_variate_theta =np.mean(Y)
mean, lower, upper = compute_ci(Y)
print(f'Estimated integral with control variate n={sample_size} is {np.round(control_variate_theta,4)}')
print(f"mean: {np.round(mean,4)}, Confidence 95%: [{np.round(lower,4)}, {np.round(upper,4)}] ")

Estimated integral with control variate n=100 is 1.7223
mean: 1.7223, Confidence 95%: [1.7099, 1.7347] 


In [6]:
# part 4 estimate integral int(0,1) (exp(x))
# using stratified sampling 
np.random.seed(42)
n = 10

U_many = []
for j in range(1,n+1):
    U_many.append(np.random.uniform((j-1)/n, j/n, n))


Y_many=[]
for U in U_many:
    Y_many.append(1/n* np.sum(np.exp(U)))

stratified_sampling_theta = np.mean(Y_many)
mean, lower, upper = compute_ci(Y_many)
print(f'Estimated integral with stratified sampling  n={sample_size} is {np.round(stratified_sampling_theta,4)}')
print(f"mean: {np.round(mean,4)}, Confidence 95%: [{np.round(lower,4)}, {np.round(upper,4)}] ")

Estimated integral with stratified sampling  n=100 is 1.7137
mean: 1.7137, Confidence 95%: [1.3447, 2.0827] 


In [7]:
# part 5 use control variable to reduce the variance of the estimator in exercise 4 (poisson arrivals)

# we have to write a discrete event simulation program for a blocking system of m service units 
# and no waiting rooms. 
# this could be like a phone help service where there is no queue, if all lines are occupied
# the phone hangs up.

# The offered traffic A is the product of the mean arrival rate and the mean service time 
import numpy as np
import RNG_mat
import scipy.stats as stats
import math

np.random.seed(42)
# ------PART 1-------
# arrival process is poison. according to notes the diffrence between points in a poisson process 
# is exponential
N_CUSTUMERS = 10_000
M_SERVERS = 10
MU_SERVICE_TIME = 8
MU_ARRIVAL_TIME = 1

N_REPS = 10




def create_poison_dist_points(n_samples=10_000):
    # if Lambda is not None:
    #     mean_ = 1/Lambda
    arrival_intervals = np.random.exponential(scale=1/1, size=n_samples)
    return arrival_intervals

def create_poison_dist_points_service( n_samples=10_000):
    # if Lambda is not None:
    #     mean_ = 1/Lambda
    arrival_intervals = np.random.exponential(scale=8, size=n_samples)
    return arrival_intervals


def compute_ci(data, alpha=0.05):
    """Calculates the 95% Confidence Interval based on sub-samples."""
    n = len(data)
    mean = np.mean(data)
    std_err = np.std(data, ddof=1) / np.sqrt(n)
    t_crit = stats.t.ppf(1 - alpha/2, df=n - 1)
    return mean, mean - t_crit * std_err, mean + t_crit * std_err

def simulate_blocking(N_REPS,N_CUSTUMERS,arrival_distribution, service_distribution):
    blocked_list = []
    arrival_times_list = []

    for rep in range(N_REPS):
        blocked_customers = 0
        arrival_intervals = arrival_distribution(n_samples=N_CUSTUMERS)
        arrival_times = np.cumsum(arrival_intervals)
        service_intervals = service_distribution( n_samples=N_CUSTUMERS)


        next_free_server = np.zeros((M_SERVERS))

        for i in range(N_CUSTUMERS):
            current_person_arrival = arrival_times[i]

            available_servers_mask = next_free_server<=current_person_arrival

            if np.any(available_servers_mask):
                idx = np.argmax(available_servers_mask) 
                next_free_server[idx] = current_person_arrival + service_intervals[i]
            else: 
                # no free cashier
                blocked_customers +=1
        
        arrival_times_list.append(np.mean(arrival_intervals))

        blocked_list.append(blocked_customers/N_CUSTUMERS)


    
    return blocked_list, arrival_times_list

blocked_list_OG, arrival_times_list = simulate_blocking(N_REPS,N_CUSTUMERS, create_poison_dist_points,create_poison_dist_points_service)
mean, lower, upper = compute_ci(blocked_list_OG)

A = MU_SERVICE_TIME* 1/MU_ARRIVAL_TIME

def calculate_erlingB(A, M_SERVERS):
    m_list = np.arange(0,M_SERVERS+1,1).astype(int)
    bot = [math.factorial(m_list[i]) for i in range (len(m_list))]
    B = (A**M_SERVERS/math.factorial(M_SERVERS)) / (np.sum(A**m_list/ (np.array(bot))))
    return B
B = calculate_erlingB(A, M_SERVERS)

# control variate

Z = np.array(arrival_times_list).ravel()
X = np.array(blocked_list_OG).ravel()
cov_ = np.cov(X,Z)[0,1]
c = -cov_/(np.var(Z))

Y = X + c*(Z-MU_ARRIVAL_TIME)
mean_control_varitate, lower_control_varitate, upper_control_varitate = compute_ci(Y)


print(f"Simulated blocking function mean: {np.round(mean,4)}, Confidence 95%: [{np.round(lower,4)}, {np.round(upper,4)}] ")
print(f"Exact Erlang-B from slides: {np.round(B,4)} ")

print(f"\nControl Variate: {np.round(mean_control_varitate,4)}, Confidence 95%: [{np.round(lower_control_varitate,4)}, {np.round(upper_control_varitate,4)}] ")

Simulated blocking function mean: 0.1233, Confidence 95%: [0.1176, 0.129] 
Exact Erlang-B from slides: 0.1217 

Control Variate: 0.123, Confidence 95%: [0.119, 0.127] 


In [8]:
# # part 6 demonstrate the effect of using common random numbers in exercise 4 when comparing:
# # - Poisson arrivals
# # - A renewal process with hyperexponential inter-arrival time
# np.random.seed(42)
# U_A = []
# U_S = []
# coins = []
# for i in range(N_REPS):
#     U_A.append(np.random.uniform(0,1,size=N_CUSTUMERS))
#     U_S.append(np.random.uniform(0,1,size=N_CUSTUMERS))
#     coins.append(np.random.rand(N_CUSTUMERS))



# def simulate_blocking(N_REPS,N_CUSTUMERS,arrival_distribution, service_distribution):
#     blocked_list = []
#     arrival_times_list = []

#     for rep in range(N_REPS):
#         blocked_customers = 0
#         coin = coins[rep]
#         U_arrival = U_A[rep]
#         U_services = U_S[rep]
        
#         if arrival_distribution==create_poison_dist_points:
#             arrival_intervals = arrival_distribution(U_arrival)
#         else: 
#             arrival_intervals = arrival_distribution(U_arrival,coin)
#         arrival_times = np.cumsum(arrival_intervals)

#         service_intervals = service_distribution(U_services)


#         next_free_server = np.zeros((M_SERVERS))

#         for i in range(N_CUSTUMERS):
#             current_person_arrival = arrival_times[i]

#             available_servers_mask = next_free_server<=current_person_arrival

#             if np.any(available_servers_mask):
#                 idx = np.argmax(available_servers_mask) 
#                 next_free_server[idx] = current_person_arrival + service_intervals[i]
#             else: 
#                 # no free cashier
#                 blocked_customers +=1

        
#         arrival_times_list.append(np.mean(arrival_intervals))

#         blocked_list.append(blocked_customers/N_CUSTUMERS)


    
#     return blocked_list, arrival_times_list


# def exp_dist(U,lam):
#     return -np.log(U)/lam
# def exp_services(U):
#     # mean service time = 8
#     return exp_dist(U, lam=1/MU_SERVICE_TIME)

# def create_poison_dist_points(U):
#     # if Lambda is not None:
#     #     mean_ = 1/Lambda
#     arrival_intervals = exp_dist(U,1)
#     return arrival_intervals

# def hyper_exp_distribution(U,RN):
#     p1, lam1, lam2 = 0.8 , 0.8333, 5.0
#     mask = RN <= p1
#     default = exp_dist(U,lam1)
#     second = exp_dist(U,lam2)
#     return np.where(mask, default, second)


# blocked_list_arrival_poisson, _ = simulate_blocking(N_REPS,N_CUSTUMERS, create_poison_dist_points,exp_services)
# mean, lower, upper = compute_ci(blocked_list_arrival_poisson)
# print(f"POISSON ARRIVAL mean: {np.round(mean,4)}, Confidence 95%: [{np.round(lower,4)}, {np.round(upper,4)}] ")

# blocked_list_hyperexp, _ = simulate_blocking(N_REPS,N_CUSTUMERS, hyper_exp_distribution,exp_services)
# mean, lower, upper = compute_ci(blocked_list_hyperexp)
# print(f"\nHYPEREXPONENTIAL ARRIVAL mean: {np.round(mean,4)}, Confidence 95%: [{np.round(lower,4)}, {np.round(upper,4)}] ")

# A = MU_SERVICE_TIME* 1/MU_ARRIVAL_TIME
# B = calculate_erlingB(A, M_SERVERS)

# print(f"\n \nExact Erlang-B from slides: {np.round(B,4)} ")

In [9]:
# part 7 estimate P(Z>a) when Z~ Ñ(0,1) using the crude monte carlo estimater 
np.random.seed(42)
sample_size = 10_000

Z = np.random.normal(0,1,size=sample_size)
PZ = np.sum(Z>0)/sample_size

# Then apply importance sampling using a normal distribution with mean a variance sigma^2
#  - Start with sigma^2 = 1
#  - Experiment with different values of a (e.g. , a=2 and a=4)
#  - Experiment with different sample sizes
#  - If time permits, investigate other values of sigma^2

#Finally, disguss the effiency of the methods



sample_size = 10_000

# from slide 13 day 4
a2 = 2
a4 = 4
s = 1


PZ2 = np.sum(Z>a2)/sample_size
PZ4 = np.sum(Z>a4)/sample_size

print(f"P(Z>{a2}) = {PZ2} , P(Z)>{a4}) = {PZ4}")

P(Z>2) = 0.0237 , P(Z)>4) = 0.0


In [10]:
np.random.seed(42)
def crude_mc(a, N,s):
    Y = np.random.normal(a, s, size=N)

    # f 
    f = stats.norm.pdf(Y, loc=0, scale=1)
    g = stats.norm.pdf(Y,loc=a, scale=s)

    weight  = f/g
    h = Y>a

    samples = h * weight
    estimate = np.mean(samples)
    std_error = np.std(samples, ddof=1) / np.sqrt(N)

    ci_low = estimate - 1.96 * std_error
    ci_high = estimate + 1.96 * std_error

    return estimate, ci_low, ci_high

Ns = np.arange(1000,100_000, 10_000)
A_s = np.arange(1,50,10) 
s=1
for N in Ns:
    for a in A_s:
        mu, ci1, ci2 = crude_mc(a, N,s)
        print(f"a={a}, N = {N},     mu: {mu} , CI [{ci1}, {ci2}]")

a=1, N = 1000,     mu: 0.16495129522522695 , CI [0.152829784717033, 0.17707280573342088]
a=11, N = 1000,     mu: 1.839170504693145e-28 , CI [1.4278776303404014e-28, 2.2504633790458885e-28]
a=21, N = 1000,     mu: 3.482352438374839e-98 , CI [2.332837950174622e-98, 4.631866926575056e-98]
a=31, N = 1000,     mu: 2.213933918089617e-211 , CI [2.213933918089617e-211, 2.213933918089617e-211]
a=41, N = 1000,     mu: 0.0 , CI [0.0, 0.0]
a=1, N = 11000,     mu: 0.1580337977310589 , CI [0.15444480736400523, 0.1616227880981126]
a=11, N = 11000,     mu: 2.0874482768446387e-28 , CI [1.9518809172578253e-28, 2.223015636431452e-28]
a=21, N = 11000,     mu: 3.189861522932876e-98 , CI [2.8876833188276233e-98, 3.492039727038129e-98]
a=31, N = 11000,     mu: 2.73157009911898e-211 , CI [2.73157009911898e-211, 2.73157009911898e-211]
a=41, N = 11000,     mu: 0.0 , CI [0.0, 0.0]
a=1, N = 21000,     mu: 0.15998002237940218 , CI [0.15737743530754894, 0.1625826094512554]
a=11, N = 21000,     mu: 1.909008240198036

In [11]:
# part 8 
# Use importance sampling with g(x) = lam*exp(-lam*x)
# to estimate int(0,1)(exp(x))

# try to indentify the optimal value of lam by analysing the variance of 
#           h(X)f(X)
#           ________
#              g(x)

sample_size = 10_000
def esitmate_variance_importance_sampling(lam):
    sample_size = 10_000
    x = np.random.exponential(scale =1/lam,size=sample_size)


    g = lam * np.exp(-lam*x)
    f = np.exp(x)

    # values 
    # x is between 0 and 1 because integral
    # h us our indicator function
    values =np.where( x<=1 , f/g, 0)

    estimate_mean = np.mean(values)
    estimate_variance = np.var(values)
    return estimate_variance


lams = np.linspace(0.01,10,1000 )
all_vars = []
best_lam = None
best_var = np.inf
for l in lams:
    var_current = esitmate_variance_importance_sampling(l)
    all_vars.append(var_current)
    if var_current< best_var:
        best_var = var_current
        best_lam = l


print(f"best lambda is {best_lam} with variance {best_var}")

/var/folders/mk/0w5j6hcs1js4172_jsztz29w0000gn/T/ipykernel_52395/4243392054.py:17: RuntimeWarning: overflow encountered in exp
  f = np.exp(x)


best lambda is 1.3800000000000001 with variance 2.946766140505915


In [12]:
# part 9
# in report